# N01 · GPU 显存解剖：从“为什么 OOM”到“该改哪个旋钮”


> 学习方式建议：先读“心智模型”，再运行代码实验，最后做练习题。每道题都附答案解析，不是为了考倒你，而是为了暴露最常见的模棱两可点。  
> 本 notebook 只做概念和可复现小实验；真实工程闭环请回到对应 lab 运行 `make smoke M=...` 并查看 `runs/.../metrics.jsonl`、日志和报告。


## 本节要解决的问题

很多自学者遇到 CUDA OOM 时，第一反应是“模型太大”。这句话往往不够工程化，因为训练时显存至少由五类东西组成：

1. **参数（parameters）**：模型权重本身。
2. **梯度（gradients）**：反向传播后每个可训练参数对应的梯度。
3. **优化器状态（optimizer states）**：Adam/AdamW 通常保存一阶、二阶动量，可能比参数还大。
4. **激活值（activations）**：forward 中为了 backward 保存的中间结果，通常随 batch、sequence length、层数增长。
5. **临时 buffer / allocator reserved memory**：kernel workspace、通信 buffer、PyTorch caching allocator 预留但未必正在被 tensor 使用的显存。

本节目标不是背公式，而是形成一个判断流程：**OOM 发生时，我怎么判断该降 batch、降 sequence、换 optimizer、开 mixed precision、开 activation checkpoint，还是检查显存碎片/外部占用？**


## 学习地图与版本说明（截至 2026-04-30）

本节建议按“三层账本”学习：先把显存拆成参数、梯度、优化器、激活、临时 buffer；再用小实验估算每一类随 batch、sequence、dtype 的变化；最后回到 L01 的 smoke run，用真实 `peak_memory_gb` 和日志判断哪个旋钮最值得改。

版本上，本教程优先参考 PyTorch stable 文档中的 CUDA semantics、CUDA memory snapshot 和 `torch.cuda.memory_stats/max_memory_allocated`。截至当前 PyTorch stable 文档，`allocated` 更接近 tensor 正在使用的显存，`reserved` 更接近 caching allocator 已向 CUDA 申请并管理的内存池；`nvidia-smi` 看到的是进程级外部视角，不能直接等同于 tensor 实占。

学完本节，你应该能完成三件事：第一，看到 OOM 报错时不再只说“模型太大”，而能判断是加载、forward、backward、optimizer step 还是评估阶段失败；第二，能说明为什么 AdamW、长序列和 activation checkpoint 对显存账本的影响完全不同；第三，能写出一个最小复现，固定 shape、固定 step、一次只改一个变量，避免“同时改五个参数但不知道哪个有效”。


## 1. 心智模型：训练显存不是一个数字，而是一张账本

你可以把 GPU 显存想象成公司预算：

- 参数显存像“固定资产”：模型一加载就占着。
- 梯度显存像“每轮训练产生的账单”：训练模式下才需要，推理通常没有。
- 优化器状态像“财务台账”：AdamW 会保存动量，训练时非常贵。
- 激活显存像“会议记录”：forward 时保存，backward 用完后释放；长序列时记录会非常厚。
- reserved memory 像“预订会议室”：PyTorch 可能先向 CUDA 申请一块大空间留着复用，所以 `nvidia-smi` 看到的显存不一定等于 tensor 真正占用。

PyTorch 官方 CUDA notes 区分了 `memory_allocated/max_memory_allocated` 与 `memory_reserved/max_memory_reserved`。工程判断中，allocated 更接近 tensor 实占，reserved 更接近 PyTorch allocator 管理的池子大小。


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BYTES = {"fp32": 4, "bf16/fp16": 2, "fp8/int8": 1}

def gb(num_bytes):
    return num_bytes / 1024**3

def training_memory_components(params_billion=1.0, precision="bf16/fp16", optimizer="adamw"):
    param_gb = gb(params_billion * 1e9 * BYTES[precision])
    grad_gb = param_gb
    if optimizer == "adamw":
        # 简化：AdamW 一阶/二阶动量常按 fp32 保存，约 8 bytes/param；实际还取决于 ZeRO/FSDP/分布式优化器。
        optim_gb = gb(params_billion * 1e9 * 8)
    elif optimizer == "sgd":
        optim_gb = gb(params_billion * 1e9 * 4)
    else:
        optim_gb = 0
    return pd.Series({"参数GB": param_gb, "梯度GB": grad_gb, "优化器GB": optim_gb, "合计GB(不含激活)": param_gb + grad_gb + optim_gb})

table = pd.DataFrame({
    "1B bf16 AdamW": training_memory_components(1, "bf16/fp16", "adamw"),
    "7B bf16 AdamW": training_memory_components(7, "bf16/fp16", "adamw"),
    "7B fp32 AdamW": training_memory_components(7, "fp32", "adamw"),
}).T

display(table.round(2))


## 2. 为什么 sequence length 常常比你想象中危险？

参数显存主要跟模型大小有关，batch/sequence 不变时比较稳定；激活显存则跟一次 forward 的张量形状强相关。对于 Transformer，很多中间张量都含有 `batch × seq_len × hidden`，attention 相关中间量还可能出现与 `seq_len²` 有关的计算或临时状态（现代 kernel 会优化保存方式，但长序列仍然昂贵）。

所以如果你把 `seq_len` 从 1024 提到 4096，显存压力不只是“变成 4 倍”这么简单；kernel 选择、临时 buffer、attention 实现、checkpointing 策略都会被改变。


In [ ]:
def rough_activation_gb(batch, seq_len, hidden=4096, layers=32, bytes_per_elem=2, saved_tensors_factor=8.0, recompute=False):
    # 这是教学估算，不等于任何框架的精确显存。saved_tensors_factor 表示每层保存的中间张量倍数。
    factor = saved_tensors_factor * (0.35 if recompute else 1.0)
    return gb(batch * seq_len * hidden * layers * bytes_per_elem * factor)

rows = []
for batch in [1, 2, 4, 8]:
    for seq_len in [512, 1024, 2048, 4096, 8192]:
        rows.append({
            "batch": batch,
            "seq_len": seq_len,
            "不开重算GB": rough_activation_gb(batch, seq_len),
            "开重算GB": rough_activation_gb(batch, seq_len, recompute=True),
        })

df = pd.DataFrame(rows)
display(df.round(2).head(10))

pivot = df[df["batch"] == 4].set_index("seq_len")[["不开重算GB", "开重算GB"]]
pivot.plot(marker="o", title="batch=4 时 sequence length 对激活显存的影响")
plt.ylabel("估算激活显存 GB")
plt.show()


## 3. allocated、reserved、nvidia-smi 三者怎么同时看？

常见痛点：`nvidia-smi` 显示用了很多显存，但 `torch.cuda.memory_allocated()` 看起来没那么多。原因可能是：

- PyTorch caching allocator 已经向 CUDA 预留了内存，准备用于后续 tensor 分配。
- CUDA context、NCCL、cuDNN、FlashAttention、第三方库也会占显存，这些不一定都被 PyTorch allocator 统计。
- 显存碎片导致“总空闲够，但连续可分配块不够”。
- 你保留了计算图引用，例如把 `loss` tensor 直接 append 到 Python list，而不是 `loss.item()`。

工程上建议同时记录：

```python
torch.cuda.max_memory_allocated()
torch.cuda.max_memory_reserved()
nvidia-smi / pynvml 外部观测
```

如果 reserved 明显大于 allocated，不要马上认为“泄漏”。先判断是不是 allocator 缓存或碎片，再看是否有 Python 引用保留计算图。


In [ ]:
# CPU 环境也能运行；有 CUDA 时会打印 PyTorch allocator 指标。
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        x = torch.empty((1024, 1024, 256), device="cuda", dtype=torch.float16)
        print("allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 3))
        print("reserved  GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
        del x
        torch.cuda.empty_cache()
        print("after empty_cache reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 3))
    else:
        print("当前环境没有 CUDA：请在 GPU 节点运行本 cell 观察 allocated/reserved。")
except Exception as exc:
    print("跳过 CUDA 观测：", exc)


## 4. OOM 排障决策树

遇到 OOM，不要先随机改参数。按下面顺序问：

1. **发生阶段**：模型加载时、forward、backward、optimizer step、eval/generation、checkpoint save？
2. **变化变量**：最近改了 batch、seq_len、precision、optimizer、TP/PP、activation checkpoint、数据格式，还是版本？
3. **显存桶**：参数/梯度/优化器/激活/临时 buffer 哪个最可疑？
4. **最小复现**：能否用 2 step、1 batch、固定 shape 复现？
5. **最小修复**：一次只改一个旋钮，例如先只降 seq_len，不要同时降 batch、换 dtype、换模型。

与本课程连接：

- L01 会真实记录 `peak_memory_gb`、forward/backward/optimizer timing。
- L05 会把 Megatron TP/PP/recompute 纳入扩展估算。
- Debug ticket：`pt_oom_001`、`mgt_oom_001`、`tt_fsdp_memory_003`。


## 5. 企业面试/工程判断痛点题（带答案）

### 题 1：一个 7B 模型用 bf16 训练，为什么“权重 14GB”不代表 24GB 卡一定能训？

**答案解析：** bf16 参数约 14GB 只是参数本身。训练还要梯度、优化器状态和激活。AdamW 的优化器状态可能几十 GB；长序列激活也可能十几 GB。因此 24GB 卡通常只能做很小 batch、短序列、LoRA/冻结部分参数，或需要 ZeRO/FSDP/TP/activation checkpointing。

### 题 2：`torch.cuda.empty_cache()` 能解决所有 OOM 吗？

**答案解析：** 不能。它主要把 PyTorch allocator 中未使用的 cached blocks 还给 CUDA，不能释放仍被 tensor 引用的显存，也不能降低当前模型本身需要的峰值显存。若 OOM 是激活峰值太高，应该调 batch/seq_len/checkpointing，而不是迷信 empty_cache。

### 题 3：如果 OOM 发生在 backward，而不是 forward，优先怀疑什么？

**答案解析：** backward 需要读取/保存的激活、产生梯度，并可能触发额外临时 buffer。优先看激活显存、梯度显存、checkpointing、sequence length、loss 是否保留计算图，而不是只看参数大小。

### 题 4：reserved 很大、allocated 不大，一定是显存泄漏吗？

**答案解析：** 不一定。PyTorch caching allocator 会预留显存复用，reserved 大可能是正常缓存或碎片。要结合趋势看：如果 allocated 随 step 单调增长，可能有引用泄漏；如果 allocated 稳定而 reserved 较大，可能只是缓存。

### 题 5：把 batch 从 4 降到 2 仍 OOM，下一步该怎么查？

**答案解析：** 先确认 OOM 阶段和峰值来源。如果 sequence length 极长，降 batch 可能还不够；如果模型加载就 OOM，batch 无关；如果 optimizer step OOM，可能是优化器状态；如果 reserved/allocated 异常，查碎片或外部占用。


## 6. 小结

本节最重要的不是公式，而是判断路径：

- 参数显存回答“模型能否加载”。
- 参数 + 梯度 + 优化器 + 激活回答“能否训练”。
- allocated/reserved/nvidia-smi 共同回答“显存到底在哪里”。
- OOM 修复要一变量一实验，不能靠玄学调参。

下一步：运行 `make smoke M=l02_pytorch_systems`，把真实 `peak_memory_gb` 和本 notebook 的估算进行对照。


## 参考资料

- PyTorch CUDA semantics / memory management: https://docs.pytorch.org/docs/stable/notes/cuda.html
- PyTorch Understanding CUDA Memory Usage: https://docs.pytorch.org/docs/stable/torch_cuda_memory.html
- PyTorch `max_memory_allocated`: https://docs.pytorch.org/docs/stable/generated/torch.cuda.max_memory_allocated.html
- PyTorch FAQ：GPU memory 与 caching allocator: https://docs.pytorch.org/docs/stable/notes/faq.html
